In [1]:
import argparse
import glob
import json
import math
import os
import re
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

In [2]:
ROOT = os.getcwd()
DEFAULT_DATA = os.path.join(ROOT, "dataverse_files")
DEFAULT_WORK = os.path.join(ROOT, "work")
DEFAULT_OUT = os.path.join(ROOT, "output")

In [3]:
PAT_ALL = [211111, 221112, 324110, 325110, 331312, 334413, 454111, 517110, 517210, 518210, 519130, 524113, 541612]
PAT_MED = [454111, 517110, 518210, 519130, 524113, 541612]
NAICS_INTERNET = [454111, 454112, 511110, 511120, 511130, 511140, 511191, 511199, 515112, 517110, 517210, 517919, 519130, 541511, 561422]
DATA_PATENT_CLASSES = ["702", "703", "700", "706", "715", "707", "710", "717", "709", "704", "341", "705", "701"]

In [4]:
COMPUSTAT_VARS = ["revtq", "atq", "cheq", "cstkq", "lseq", "ltq", "seqq"]
SHORT_VARS = ["rev", "at", "che", "cstk", "lse", "lt", "seq"]

BLOCKS_AS_WRITTEN = [["at"], ["rev", "lt"]]
BLOCKS_INTENDED = [["at"], ["rev"], ["lt"]]

In [5]:
def open_log(path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    handle = open(path, "w")

    def say(*parts):
        line = " ".join(str(p) for p in parts)
        print(line, flush=True)
        handle.write(line + "\n")
        handle.flush()
    return say

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=1, default=str)

In [6]:
def digamma(x):
    x = np.asarray(x, dtype=float).copy()
    res = np.zeros_like(x)
    while True:
        small = x < 6
        if not small.any():
            break
        res[small] -= 1.0 / x[small]
        x[small] += 1
    f = 1.0 / (x * x)
    res += (np.log(x) - 0.5 / x
            - f * (1/12 - f * (1/120 - f * (1/252 - f * (1/240 - f / 132)))))
    return res


def trigamma(x):
    x = np.asarray(x, dtype=float).copy()
    res = np.zeros_like(x)
    while True:
        small = x < 6
        if not small.any():
            break
        res[small] += 1.0 / (x[small] ** 2)
        x[small] += 1
    f = 1.0 / (x * x)
    res += (1 / x + f / 2
            + f / x * (1/6 - f * (1/30 - f * (1/42 - f * (1/30 - f * 5 / 66)))))
    return res


def normal_p_two_sided(z):
    return math.erfc(abs(z) / math.sqrt(2))


def _beta_continued_fraction(a, b, x):
    MAXIT, EPS, FPMIN = 300, 3e-14, 1e-300
    qab, qap, qam = a + b, a + 1, a - 1
    c, d = 1.0, 1 - qab * x / qap
    d = 1 / (d if abs(d) > FPMIN else FPMIN)
    h = d
    for m in range(1, MAXIT + 1):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1 + aa * d; d = d if abs(d) > FPMIN else FPMIN
        c = 1 + aa / c; c = c if abs(c) > FPMIN else FPMIN
        d = 1 / d; h *= d * c
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1 + aa * d; d = d if abs(d) > FPMIN else FPMIN
        c = 1 + aa / c; c = c if abs(c) > FPMIN else FPMIN
        d = 1 / d; de = d * c; h *= de
        if abs(de - 1) < EPS:
            break
    return h


def incomplete_beta(a, b, x):
    if x <= 0:
        return 0.0
    if x >= 1:
        return 1.0
    lbt = (math.lgamma(a + b) - math.lgamma(a) - math.lgamma(b)
           + a * math.log(x) + b * math.log(1 - x))
    if x < (a + 1) / (a + b + 2):
        return math.exp(lbt) * _beta_continued_fraction(a, b, x) / a
    return 1 - math.exp(lbt) * _beta_continued_fraction(b, a, 1 - x) / b


def t_p_two_sided(t, df):
    return incomplete_beta(df / 2, 0.5, df / (df + t * t))

In [7]:
def factor_codes(labels):
    c, _ = pd.factorize(pd.Series(labels), use_na_sentinel=False)
    return c.astype(np.int64)


def demean(M, fe_list, w=None, tol=1e-10, maxit=10000):
    M = np.array(M, dtype=float, copy=True)

    if M.ndim == 1:
        M = M[:, None]

    if w is None:
        w = np.ones(M.shape[0])

    group_weights = [np.bincount(g, weights=w) for g in fe_list]

    for _ in range(maxit):
        delta = 0.0

        for g, s in zip(fe_list, group_weights):
            for j in range(M.shape[1]):
                m = np.bincount(g, weights=w * M[:, j]) / s
                M[:, j] -= m[g]
                delta = max(delta, np.abs(m).max())

        if len(fe_list) == 1 or delta < tol:
            break

    return M


def is_nested(fe, cl):
    first = np.full(fe.max() + 1, -1)
    first[fe] = cl
    return bool(np.all(first[fe] == cl))


def count_fe_params(fe_codes, cl):
    return sum(len(np.unique(f)) - (1 if i else 0) for i, f in enumerate(fe_codes) if not is_nested(f, cl))


def cluster_vcov(Xt, scores, bread, cl, n, K):
    G = cl.max() + 1
    S = np.zeros((G, Xt.shape[1]))

    for j in range(Xt.shape[1]):
        S[:, j] = np.bincount(cl, weights=Xt[:, j] * scores, minlength=G)

    adj = G / (G - 1) * (n - 1) / (n - K)

    return adj * bread @ (S.T @ S) @ bread, G


def make_result(names, coef, vcov, n, G, dist, extra=None):
    se = np.sqrt(np.diag(vcov))
    stat = coef / se

    if dist == "t":
        p = np.array([t_p_two_sided(s, G - 1) for s in stat])
    else:
        p = np.array([normal_p_two_sided(s) for s in stat])

    return {"names": list(names), "coef": coef, "se": se, "p": p, "n": n, "G": G, "extra": extra or {}}


def format_result(r, irr=False):

    lines = [f"{'term':<22}{'coef':>11}{'se':>10}{'p':>9}" + ("   IRR" if irr else "")]

    for k, name in enumerate(r["names"]):
        p = r["p"][k]
        star = "***" if p < .01 else "**" if p < .05 else "*" if p < .1 else ""
        line = f"{name:<22}{r['coef'][k]:>11.4f}{r['se'][k]:>10.4f}{p:>9.4f} {star:<3}"

        if irr:
            line += f"  {math.exp(r['coef'][k]):.4f}"

        lines.append(line)

    lines.append(f"N = {r['n']:,}   clusters = {r['G']}" + "".join(f"   {k} = {v}" for k, v in r["extra"].items()))

    return "\n".join(lines)


def result_to_dict(r):
    out = {name: dict(coef=float(r["coef"][k]), se=float(r["se"][k]), p=float(r["p"][k]), irr=float(math.exp(r["coef"][k]))) for k, name in enumerate(r["names"])}

    return out | {"N": r["n"], "G": r["G"], **r["extra"]}


def drop_collinear(Xt, names, tol=1e-7):
    G = Xt.T @ Xt
    d = np.sqrt(np.clip(np.diag(G), 0, None))
    keep = []

    for j in range(Xt.shape[1]):
        if d[j] == 0:
            continue

        if keep:
            Gkk = G[np.ix_(keep, keep)] / np.outer(d[keep], d[keep])
            gkj = G[keep, j] / (d[keep] * d[j])
            resid2 = 1 - gkj @ np.linalg.lstsq(Gkk, gkj, rcond=None)[0]
        else:
            resid2 = 1.0

        if resid2 > tol ** 2:
            keep.append(j)

    dropped = [names[j] for j in range(len(names)) if j not in keep]

    return keep, dropped


def ols_fe(y, X, names, fe, cluster):
    fe_codes = [factor_codes(f) for f in fe]

    cl = factor_codes(cluster)
    Xt = demean(X, fe_codes)
    yt = demean(y, fe_codes)[:, 0]

    keep, dropped = drop_collinear(Xt, names)

    Xt = Xt[:, keep]

    names = [names[j] for j in keep]

    bread = np.linalg.inv(Xt.T @ Xt)

    b = bread @ Xt.T @ yt

    resid = yt - Xt @ b
    n = len(y)
    K = len(b) + count_fe_params(fe_codes, cl)
    V, G = cluster_vcov(Xt, resid, bread, cl, n, K)

    return make_result(names, b, V, n, G, "t", {"dropped": dropped} if dropped else None)


def drop_all_zero_groups(y, fe_codes):
    keep = np.ones(len(y), bool)
    changed = True

    while changed:
        changed = False
        for g in fe_codes:
            total = np.bincount(g[keep], weights=y[keep], minlength=g.max() + 1)
            bad = keep & (total[g] == 0)
            if bad.any():
                keep &= ~bad
                changed = True

    return keep


def nb_loglik(y, mu, theta):
    lg = np.vectorize(math.lgamma)
    uy = np.unique(y)
    lg_lookup = lg(uy + theta)
    lg_y_theta = lg_lookup[np.searchsorted(uy, y)]
    return np.sum(lg_y_theta - math.lgamma(theta) + theta * np.log(theta / (theta + mu)) + y * np.log(np.where(y > 0, mu / (theta + mu), 1)))


def nb_theta_ml(y, mu, theta0, maxit=100):
    lt = math.log(max(theta0, 1e-6))

    for _ in range(maxit):
        t = math.exp(lt)
        s = np.sum(digamma(y + t) - digamma(t) + np.log(t / (t + mu)) + 1 - (y + t) / (t + mu))
        h = np.sum(trigamma(y + t) - trigamma(t) + 1 / t - 2 / (t + mu) + (y + t) / (t + mu) ** 2)

        g1 = s * t
        g2 = h * t * t + s * t

        step = g1 / g2 if g2 < 0 else -0.5 * np.sign(g1)
        step = max(min(step, 2), -2)
        lt -= step

        if abs(step) < 1e-10:
            break
        
    return math.exp(lt)


def nb_joint_vcov(y, X, mu, theta, fe_codes, cl, n, K):
    t = theta
    w = t * (y + t) * mu / (t + mu) ** 2
    s = t * (y - mu) / (t + mu)
    c = mu * (y - mu) / (t + mu) ** 2

    s_theta = digamma(y + t) - digamma(t) + np.log(t / (t + mu)) + 1 - (y + t) / (t + mu)
    h_theta = trigamma(y + t) - trigamma(t) + 1 / t - 2 / (t + mu) + (y + t) / (t + mu) ** 2
    r = c / w

    both = demean(np.column_stack([X, r]), fe_codes, w)
    Xt, rt = both[:, :-1], both[:, -1]
    p = X.shape[1]

    H = np.zeros((p + 1, p + 1))
    H[:p, :p] = (Xt.T * w) @ Xt
    H[:p, p] = H[p, :p] = -(Xt.T @ c)
    H[p, p] = -(h_theta.sum() + np.sum(c * r) - np.sum(w * rt ** 2))

    bread = np.linalg.inv(H)

    psi_theta = s_theta + (r - rt) * s

    Z = np.column_stack([Xt * s[:, None], psi_theta])
    G = cl.max() + 1
    S = np.column_stack([np.bincount(cl, weights=Z[:, j], minlength=G) for j in range(p + 1)])
    adj = G / (G - 1) * (n - 1) / (n - K)
    V = adj * bread @ (S.T @ S) @ bread
    
    return V[:p, :p], G

In [8]:
def count_fe(y, X, names, fe, cluster, family="poisson", tol=1e-8, maxit=200, nuisance=0):
    y = np.asarray(y, float)
    X = np.asarray(X, float)

    fe_codes = [factor_codes(f) for f in fe]
    cl = factor_codes(cluster)
    keep = drop_all_zero_groups(y, fe_codes)
    n_removed = int((~keep).sum())

    y, X, cl = y[keep], X[keep], cl[keep]

    fe_codes = [factor_codes(g[keep]) for g in fe_codes]
    cl = factor_codes(cl)
    n = len(y)

    theta = 1.0
    mu = (y + y.mean()) / 2
    eta = np.log(mu)
    dev_old = np.inf

    for it in range(maxit):
        w = mu / (1 + mu / theta) if family == "negbin" else mu.copy()
        z = eta + (y - mu) / mu
        Zt = demean(np.column_stack([z, X]), fe_codes, w)
        zt, Xt = Zt[:, 0], Zt[:, 1:]

        if it == 0:
            kept_cols, dropped = drop_collinear(Xt * np.sqrt(w)[:, None], names)
            n_nuisance_kept = sum(1 for j in kept_cols if j >= X.shape[1] - nuisance)
            X, Xt = X[:, kept_cols], Xt[:, kept_cols]
            names = [names[j] for j in kept_cols]

        XtW = Xt.T * w

        b = np.linalg.solve(XtW @ Xt, XtW @ zt)
        eta = z - (zt - Xt @ b)
        mu = np.exp(eta)

        if family == "negbin":
            theta = nb_theta_ml(y, mu, theta)
            dev = -nb_loglik(y, mu, theta)
        else:
            dev = -np.sum(y * np.log(np.where(y > 0, mu, 1)) - mu)
        if abs(dev - dev_old) / (abs(dev) + 0.1) < tol:
            break
        dev_old = dev

    K = len(b) - n_nuisance_kept + count_fe_params(fe_codes, cl)

    if family == "negbin":
        V, G = nb_joint_vcov(y, X, mu, theta, fe_codes, cl, n, K + 1)
    else:
        Xt = demean(X, fe_codes, mu)
        bread = np.linalg.inv((Xt.T * mu) @ Xt)
        V, G = cluster_vcov(Xt, y - mu, bread, cl, n, K)

    extra = {"iter": it + 1}

    if family == "negbin":
        extra["theta"] = round(float(theta), 4)
    if n_removed:
        extra["removed_allzero"] = n_removed
    if dropped:
        extra["dropped"] = dropped
        
    p = len(b) - n_nuisance_kept

    return make_result(names[:p], b[:p], V[:p, :p], n, G, "z", extra)

In [9]:
def balanced_panel(df, unit, time_col, cols):
    units = np.sort(df[unit].unique())
    T = int(df[time_col].max())

    ui = np.searchsorted(units, df[unit].to_numpy())
    ti = df[time_col].to_numpy().astype(int) - 1

    arrays = {}

    for c in cols:
        a = np.full((len(units), T), np.nan)
        a[ui, ti] = df[c].to_numpy(float)
        arrays[c] = a

    return {"units": units, "T": T, "tmin": int(df[time_col].min()), "arr": arrays}


def find_treated(D, lag, tmin):
    prev = np.full_like(D, np.nan)
    prev[:, 1:] = D[:, :-1]
    ui, ti = np.where((D == 1) & (prev == 0))
    keep = (ti + 1 - lag) >= tmin
    return ui[keep], ti[keep]


def matched_sets(D, ui, ti, lag):
    Dm = np.where(np.isnan(D), -1.0, D)
    sets = []

    for u, t in zip(ui, ti):
        hist = Dm[u, t - lag:t + 1].copy()
        hist[-1] = 0
        ok = np.all(Dm[:, t - lag:t + 1] == hist, axis=1)
        ok[u] = False
        sets.append(np.where(ok)[0])

    return sets


def clean_leads(Y, ui, ti, sets, max_lead):
    T = Y.shape[1]

    out = []

    for u, t, s in zip(ui, ti, sets):
        lo, hi = t - 1, t + max_lead
        if lo < 0 or hi >= T or np.isnan(Y[u, lo:hi + 1]).any():
            continue

        s = s[~np.isnan(Y[s, lo:hi + 1]).any(axis=1)]

        if len(s):
            out.append((u, t, s))

    return out


def lagged_covariates(panel, blocks, lag, t):
    cols = []
    n_units = len(panel["units"])

    for block in blocks:
        for k in range(1, lag + 1):
            v = 0.0
            for var in block:
                col = np.full(n_units, np.nan) if t - k < 0 else panel["arr"][var][:, t - k]
                v = v + col
            cols.append(v)

    X = np.column_stack(cols)
    na = np.isnan(X).astype(float)

    return np.column_stack([np.nan_to_num(X, nan=0.0), na])


def drop_constant_duplicate_cols(M):
    keep = []

    for j in range(M.shape[1]):
        if np.all(M[:, j] == M[0, j]):
            continue

        if any(np.array_equal(M[:, j], M[:, k]) for k in keep):
            continue
        
        keep.append(j)

    return keep


def closest_weights(d, size, ties_round=False, positive_only=False):
    w = np.zeros(len(d))
    cand = (d > 0) if positive_only else np.ones(len(d), bool)
    dc = d[cand]

    if len(dc) < size:
        w[cand] = 1 / len(dc) if len(dc) else 0
        return w
    
    cutoff = np.sort(dc)[:size + 1].max()
    dd = np.round(d, 5) if ties_round else d
    cutoff = round(cutoff, 5) if ties_round else cutoff

    if np.sum((dd < cutoff) & cand) < size:
        sel = (dd <= cutoff) & cand
        w[sel] = 1 / sel.sum()
    else:
        sel = (dd < cutoff) & cand
        w[sel] = 1 / size

    return w


def mahalanobis_weights(panel, blocks, lag, matched, size):
    out, cache = [], {}

    for u, t, s in matched:
        if len(s) == 1:
            out.append(np.array([1.0]))
            continue

        if t not in cache:
            cache[t] = lagged_covariates(panel, blocks, lag, t)

        X = cache[t]
        C, x0 = X[s], X[u]

        if len(s) < size:
            out.append(np.full(len(s), 1 / len(s)))
            continue

        S = np.cov(C, rowvar=False)

        if abs(np.linalg.det(S)) <= 1e-5:
            keep = drop_constant_duplicate_cols(C)
            if 0 < len(keep) < C.shape[1]:
                C, x0 = C[:, keep], x0[keep]
                S = np.cov(C, rowvar=False)
        try:
            Si = np.linalg.inv(S)
        except np.linalg.LinAlgError:
            Si = np.linalg.pinv(S)

        diff = C - x0

        d = np.einsum("ij,jk,ik->i", diff, Si, diff)
        out.append(closest_weights(d, size, ties_round=True))

    return out

In [10]:
def fit_logit(X, y, maxit=100):
    Xc = np.column_stack([np.ones(len(X)), X])
    b = np.zeros(Xc.shape[1])

    for _ in range(maxit):
        p = 1 / (1 + np.exp(-(Xc @ b)))
        W = np.clip(p * (1 - p), 1e-12, None)
        step = np.linalg.lstsq((Xc.T * W) @ Xc, Xc.T @ (y - p), rcond=None)[0]

        b += step

        if np.abs(step).max() < 1e-10:
            break

    return b


def fit_cbps_exact(X, y, b0, maxit=200):
    Xc = np.column_stack([np.ones(len(X)), X])
    b = b0.copy()
    for _ in range(maxit):
        p = 1 / (1 + np.exp(-(Xc @ b)))
        r = p / (1 - p)
        g = Xc.T @ (y - (1 - y) * r)
        J = -(Xc.T * ((1 - y) * r)) @ Xc
        step = np.linalg.lstsq(J, -g, rcond=None)[0]
        b += step
        if np.abs(step).max() < 1e-10:
            break
    return b


def propensity_weights(panel, blocks, lag, matched, size, method):
    rows, cache = [], {}

    for u, t, s in matched:
        if t not in cache:
            cache[t] = lagged_covariates(panel, blocks, lag, t)
        X = cache[t]

        rows.append(np.column_stack([s, X[s], np.zeros(len(s))]))
        rows.append(np.column_stack([[u], X[[u]], np.ones(1)]))

    pooled = np.unique(np.vstack(rows), axis=0)
    Xp, yp = pooled[:, 1:-1], pooled[:, -1]
    keep = drop_constant_duplicate_cols(Xp)
    b = fit_logit(Xp[:, keep], yp)

    if method.startswith("CBPS"):
        b = fit_cbps_exact(Xp[:, keep], yp, b)

    def score(Z):
        return 1 - 1 / (1 + np.exp(np.column_stack([np.ones(len(Z)), Z]) @ b))

    out = []

    for u, t, s in matched:
        if len(s) == 1:
            out.append(np.array([1.0]))
            continue
        X = cache[t][:, keep]
        pc, pt = score(X[s]), score(X[[u]])[0]
        if method.endswith("weight"):
            r = pc / (1 - pc)
            out.append(np.full(len(s), 1 / len(s)) if r.sum() == 0 else r / r.sum())
        else:
            out.append(closest_weights(np.abs(pt - pc), size, positive_only=True))
    return out

In [11]:
def panel_match(df, unit, time_col, treatment, outcome, blocks, lag=10, lead=range(11), refinement="mahalanobis", size_match=4003):
    covariate_vars = sorted({v for b in blocks for v in b})

    panel = balanced_panel(df, unit, time_col, [treatment, outcome] + covariate_vars)

    D, Y = panel["arr"][treatment], panel["arr"][outcome]
    ui, ti = find_treated(D, lag, panel["tmin"])
    sets = matched_sets(D, ui, ti, lag)
    matched = clean_leads(Y, ui, ti, sets, max(lead))

    if refinement == "mahalanobis":
        weights = mahalanobis_weights(panel, blocks, lag, matched, size_match)
    else:
        weights = propensity_weights(panel, blocks, lag, matched, size_match, refinement)

    return {"Y": Y, "matched": matched, "weights": weights, "lead": list(lead), 
            "n_treated_obs_raw": len(ui), "treated_times": sorted({int(t) + 1 for _, t, _ in matched}), "set_sizes": [len(s) for _, _, s in matched]}


def panel_estimate(m, n_boot=1000, seed=2023, conf=0.95):
    Y, leads = m["Y"], m["lead"]
    units = sorted({int(u) for u, _, _ in m["matched"]} | {int(j) for _, _, s in m["matched"] for j in s})
    pos = {u: k for k, u in enumerate(units)}

    W = np.zeros((len(units), len(leads)))
    D = np.zeros(len(units))

    for (u, t, s), w in zip(m["matched"], m["weights"]):
        for f, F in enumerate(leads):
            W[pos[u], f] += Y[u, t + F] - Y[u, t - 1]
            np.add.at(W[:, f], [pos[j] for j in s], -w * (Y[s, t + F] - Y[s, t - 1]))
        D[pos[u]] += 1

    est = W.sum(0) / D.sum()
    rng = np.random.default_rng(seed)

    n = len(units)
    treated = D > 0
    boots = np.empty((n_boot, len(leads)))

    for k in range(n_boot):
        while True:
            draw = rng.integers(0, n, n)
            if treated[draw].any():
                break
        freq = np.bincount(draw, minlength=n)
        boots[k] = (freq @ W) / (freq @ D)

    a = (1 - conf) / 2

    return {"lead": list(leads), "estimate": est, "se": boots.std(0, ddof=1), "ci_lo": np.quantile(boots, a, axis=0), 
            "ci_hi": np.quantile(boots, 1 - a, axis=0), "boots": boots, "n_treated": int(D.sum()), "n_units": n}

In [12]:
def prep_knowledge_intensity(data, work, say):
    frames = [pd.read_excel(f, usecols=["Research Areas", "Cited Reference Count", "Number of Pages"]) for f in sorted(glob.glob(os.path.join(data, "RPP2010", "*.xls")))]
    raw = pd.concat(frames, ignore_index=True)
    say("RPP rows (R: nrow(rpp)):", len(raw))

    g = raw.dropna(subset=["Research Areas"]).groupby("Research Areas")

    rpp = pd.DataFrame({
        "refs": g["Cited Reference Count"].sum(min_count=1).where(~g["Cited Reference Count"].apply(lambda s: s.isna().any())),
        "pages": g["Number of Pages"].sum(min_count=1).where(~g["Number of Pages"].apply(lambda s: s.isna().any())),
    })

    rpp["rp"] = rpp["refs"] / rpp["pages"]
    rpp.index.name = "area"

    rpp.to_csv(os.path.join(work, "rpp.csv"))
    say(f"RPP disciplines: {len(rpp)}")
    say(rpp.to_string())

    return rpp


def read_wos_file(path, sep):
    area = re.sub(r"\.(txt|csv)$", "", os.path.basename(path))

    d = pd.read_csv(path, sep=sep, usecols=["AU", "SO", "PY", "TC"], dtype=str, keep_default_na=False, quotechar='"', on_bad_lines="warn", engine="c", encoding="utf-8", encoding_errors="replace")
    
    n_raw = len(d)

    d["PY"] = pd.to_numeric(d["PY"], errors="coerce")
    d["TC"] = pd.to_numeric(d["TC"], errors="coerce")

    d = d[d["PY"].isin(range(2011, 2021))].copy()

    d["area"] = area
    d["auno"] = d["AU"].str.count(";") + 1

    return d.drop(columns="AU"), n_raw


def prep_articles(data, work, folders, out_name, say):
    parts = []

    for sub, sep in folders:
        pattern = "*.txt" if sep == "\t" else "*.csv"

        for f in sorted(glob.glob(os.path.join(data, sub, pattern))):
            d, n_raw = read_wos_file(f, sep)
            say(f"  {sub}/{os.path.basename(f)}: raw {n_raw:,}, 2011-20 kept {len(d):,}, "f"TC missing {d['TC'].isna().sum()}")
            parts.append(d)

    a = pd.concat(parts, ignore_index=True)
    a.to_csv(os.path.join(work, out_name), index=False)

    say(f"{out_name}: {len(a):,} rows, {a['area'].nunique()} disciplines")

    return a


def prep_exchange_rates(data, work, say):
    raw = pd.read_csv(os.path.join(data, "exchange_rates.csv"), dtype=str, keep_default_na=False)
    cols = {"date": raw.columns[0]}

    for key, prefix in [("ca", "CANADA"), ("cn", "CHINA"), ("sg", "SINGAPORE"), ("hk", "HONG KONG")]:
        cols[key] = [c for c in raw.columns if c.upper().startswith(prefix)][0]

    fx = raw.iloc[5:12073][list(cols.values())].copy()
    fx.columns = list(cols.keys())

    for k in ["ca", "cn", "sg", "hk"]:
        fx[k] = pd.to_numeric(fx[k], errors="coerce")
    fx.to_csv(os.path.join(work, "fx.csv"), index=False)
    say("FX rows", len(fx), "from", fx["date"].iloc[0], "to", fx["date"].iloc[-1])

    return fx


def prep_compustat_file(data, work, source, fx, say):
    fname = {"ss": "ss.csv", "hk": "hk8719.csv", "na": "na.csv"}[source]
    use = ["gvkey", "datadate", "curcdq", "loc", "naics"] + COMPUSTAT_VARS
    parts, n_raw = [], 0

    for chunk in pd.read_csv(os.path.join(data, fname), usecols=use, dtype=str, chunksize=10_000):
        n_raw += len(chunk)
        parts.append(chunk[chunk["loc"].isin(["CHN", "USA"]) & (chunk["datadate"] > "19991231")])

    d = pd.concat(parts, ignore_index=True)

    d["naics"] = pd.to_numeric(d["naics"], errors="coerce")

    for v in COMPUSTAT_VARS:
        d[v] = pd.to_numeric(d[v], errors="coerce")
    d["date"] = pd.to_datetime(d["datadate"], format="%Y%m%d", errors="coerce").dt.strftime("%Y-%m-%d")
    d = d.merge(fx, on="date", how="inner")
    usd = d["curcdq"] == "USD"

    for v, short in zip(COMPUSTAT_VARS, SHORT_VARS):
        if source == "ss":
            d[short] = np.where(usd, d[v], d[v] / d["cn"])
        elif source == "na":
            d[short] = np.where(usd, d[v], d[v] / d["ca"])
        else:
            d[short] = np.where(usd, d[v], np.where(d["curcdq"] == "CNY", d[v] / d["cn"], np.where(d["curcdq"] == "HKD", d[v] / d["hk"], d[v] / d["sg"])))

    d["gvkey"] = d["gvkey"].astype(int)

    out = d[["date"] + SHORT_VARS + ["naics", "gvkey", "loc"]]
    say(f"{fname}: raw {n_raw:,}, after FX date merge {len(out):,}, max date {out['date'].max()}")

    return out


def prep_firm_panel(data, work, say):
    fx = prep_exchange_rates(data, work, say)
    fx = pd.read_csv(os.path.join(work, "fx.csv"), dtype={"date": str})   # same types as R
    parts = []

    for source in ["ss", "hk", "na"]:
        x = prep_compustat_file(data, work, source, fx, say)
        x = x.astype({"date": str}).sort_values("date", kind="stable")    # R merge() sorts by date
        parts.append(x)

    d = pd.concat(parts, ignore_index=True)
    d = d[(d["date"] > "1999-12-31") & d["loc"].isin(["CHN", "USA"])]
    d.to_csv(os.path.join(work, "firm_panel.csv"), index=False)

    say("firm_panel:", len(d), d.groupby("loc")["gvkey"].nunique().to_dict(), "dates", d["date"].min(), d["date"].max())

    return d


def prep_patent_intensity(data, work, say):
    kpss = pd.read_csv(os.path.join(data, "patents_xi.csv"), usecols=["permno", "class"], dtype={"class": str})

    kpss["data"] = kpss["class"].isin(DATA_PATENT_CLASSES).astype(int)
    kpss["PERMNO"] = kpss["permno"]

    stocks = []

    for f in ["NASDAQ.csv", "NYSE.csv"]:
        s = pd.read_csv(os.path.join(data, f), usecols=["PERMNO", "date", "NAICS"], dtype=str)
        stocks.append(s[s["date"] == "20090930"])

    stock09 = pd.concat(stocks, ignore_index=True)
    stock09["PERMNO"] = pd.to_numeric(stock09["PERMNO"], errors="coerce")
    stock09["NAICS"] = pd.to_numeric(stock09["NAICS"], errors="coerce")

    d5 = kpss.merge(stock09, on="PERMNO", how="outer")

    data_rows = d5[d5["data"] == 1]
    data_counts = data_rows["NAICS"].value_counts()
    industries = sorted(data_rows["NAICS"].dropna().unique())
    all_counts = d5[d5["NAICS"].isin(industries)]["NAICS"].value_counts()

    t = pd.DataFrame({"data_patents": data_counts, "all_patents": all_counts}).dropna()

    t["pct"] = t["data_patents"] / t["all_patents"]
    t.index = t.index.astype(int)
    t.index.name = "NAICS"

    t = t.sort_index()

    t.to_csv(os.path.join(work, "tableA1_patent_data_intensity.csv"))

    say(t.to_string())
    say("median pct:", t["pct"].median())

    return t

In [13]:
def run_prep(data, work, out):

    os.makedirs(work, exist_ok=True)

    say = open_log(os.path.join(work, "prep_log.txt"))
    say("== Knowledge intensity and articles")
    rpp = prep_knowledge_intensity(data, work, say)
    say("\nChinese articles")
    cn = prep_articles(data, work, [("CN0020", "\t"), ("CN1120new", ",")], "cn_articles.csv", say)
    say("\nUS articles")
    prep_articles(data, work, [("US1120", "\t")], "us_articles.csv", say)
    say("\nCN disciplines without an RPP match:", sorted(set(cn["area"]) - set(rpp.index)))
    say("\n== Firm panel")
    prep_firm_panel(data, work, say)
    say("\n== Table A1")
    prep_patent_intensity(data, work, say)

## Results

In [14]:
def figure1_search_engines(data, out):
    s = pd.read_csv(os.path.join(data, "CNsearch.csv"))
    s = s[s["Date"].notna() & s["Date"].astype(str).str.match(r"\d{4}-\d{2}")]

    year = s["Date"].str[:4].astype(int) + (s["Date"].str[5:7].astype(int) - 1) / 12

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(year, s["Google"], color="blue", lw=3)
    ax.plot(year, s["Baidu"], color="red", lw=3)
    disruptions = [(2009 + 2/12, 1), (2009 + 5/12, 0), (2009 + 6/12, 1), (2009 + 9/12, 1),
                   (2012 + 3/12, 0), (2012 + 10/12, 0), (2014, 0), (2014 + 5/12, 1),
                   (2014 + 11/12, 1), (2010 + 2/12, 0)]
    
    for x, thick in disruptions:
        ax.axvline(x, ls="--", color="k", lw=2.5 if thick else 1)

    ax.text(2013.5, 15, "Google", fontsize=14)
    ax.text(2013.5, 82, "Baidu", fontsize=14)
    ax.set_ylabel("Relative share (%)", fontsize=14)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "fig1_google_baidu.png"), dpi=150)
    plt.close(fig)


def figure3_internet_control(data, out, say):
    g = pd.read_csv(os.path.join(data, "gfwfinal.csv"))
    dates = pd.to_datetime(g["date"], format="%m/%d/%y")
    m = g.iloc[:, 1:101].interpolate(method="linear", axis=0, limit_direction="both")

    order = np.argsort(dates.to_numpy())
    M = m.to_numpy()[order].T
    d_sorted = dates.to_numpy()[order]

    cmap = LinearSegmentedColormap.from_list("rg", ["red", "green"])
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(M, aspect="auto", cmap=cmap, interpolation="nearest", vmin=np.nanmin(M), vmax=np.nanmax(M))

    years = pd.DatetimeIndex(d_sorted).year
    ticks = [np.where(years == y)[0][0] for y in sorted(set(years))]

    ax.set_xticks(ticks)
    ax.set_xticklabels(sorted(set(years)))
    ax.set_yticks([])
    ax.set_ylabel("Top-100 domains")
    ax.set_xlabel("GreatFire test date")
    fig.tight_layout()
    fig.savefig(os.path.join(out, "fig3_internet_control.png"), dpi=150)
    plt.close(fig)

    acc = pd.Series(M.mean(0), index=pd.DatetimeIndex(d_sorted)).resample("ME").mean()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(acc.index, acc.values, color="k")
    ax.axvspan(pd.Timestamp("2014-06-01"), pd.Timestamp("2016-06-30"), color="red", alpha=.15)
    ax.set_ylabel("Mean accessibility, top-100 domains")
    fig.tight_layout()
    fig.savefig(os.path.join(out, "fig3b_mean_accessibility.png"), dpi=150)
    plt.close(fig)

    say("Fig 3: test dates", len(dates), "from", d_sorted[0], "to", d_sorted[-1], "| raw non-missing share", round(float(g.iloc[:, 1:101].notna().mean().mean()), 3))
    say(acc.resample("YE").mean().round(1).to_string())

In [15]:
def run_figures(data, work, out):
    out = os.path.join(out, "figures")
    os.makedirs(out, exist_ok=True)
    say = open_log(os.path.join(out, "figures_log.txt"))
    figure1_search_engines(data, out)
    figure3_internet_control(data, out, say)

In [16]:
def chinese_firm_quarters(work):
    d = pd.read_csv(os.path.join(work, "firm_panel.csv"), dtype={"date": str, "loc": str})
    c = d[d["loc"] == "CHN"].copy()
    c = c[c["date"].str.contains("03-31|06-30|09-30|12-31")]
    c = c[c["date"] != "2010-12-31"]

    naics = c["naics"]

    c["t1"] = (naics.isin(PAT_ALL) & (c["date"] > "2014-05-31")).astype(float)
    c["t2"] = (naics.isin(PAT_MED) & (c["date"] > "2014-05-31")).astype(float)
    c["t5"] = (naics.isin(NAICS_INTERNET) & (c["date"] > "2014-05-31")).astype(float)
    c = c[~c.duplicated(["date", "gvkey"])].copy()
    c.loc[c["rev"] < 0, "rev"] = np.nan
    c.loc[c["lt"] < 0, "lt"] = np.nan

    for v in ["rev", "at", "lt"]:
        c[v] = np.log(c[v] + 0.001)

    dates = pd.unique(c["date"])

    assert len(dates) == 48, len(dates)

    c["Date"] = c["date"].map(dict(zip(dates, range(1, 49))))
    c["p1"] = (naics.isin(PAT_ALL) & (c["date"] > "2008-06-29")).astype(float)
    c["p2"] = (naics.isin(PAT_MED) & (c["date"] > "2008-06-29")).astype(float)
    c["t1p"] = (naics.isin(PAT_ALL) & (c["date"] > "2012-02-28")).astype(float)

    return c, list(dates)


def estimate_att(c, treatment, label, refinement="mahalanobis", size=4003, blocks=BLOCKS_AS_WRITTEN, n_boot=1000, seed=2023):
    start = time.time()

    m = panel_match(c, "gvkey", "Date", treatment, "rev", blocks, lag=10, lead=range(11), refinement=refinement, size_match=size)
    e = panel_estimate(m, n_boot=n_boot, seed=seed)
    e.update(label=label, treated_times=m["treated_times"], set_sizes=[min(m["set_sizes"]), max(m["set_sizes"])], n_treated_raw=m["n_treated_obs_raw"], secs=round(time.time() - start, 1))
    
    return e


def plot_att(e, path, title, ylim=None, xlab="Quarters since treatment administration"):
    fig, ax = plt.subplots(figsize=(7, 5))

    x = np.arange(len(e["lead"]))

    ax.scatter(x, e["estimate"], s=40, color="k", zorder=3)
    ax.vlines(x, e["ci_lo"], e["ci_hi"], color="k")
    ax.axhline(0, ls="--", color="k")
    ax.set_xticks(x)
    ax.set_xticklabels([f"t+{l}" for l in e["lead"]])
    ax.set_xlabel(xlab)
    ax.set_ylabel("Estimated treatment effect on revenue")

    if ylim:
        ax.set_ylim(*ylim)

    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def format_att(e):
    lines = [f"--- {e['label']}  (treated obs {e['n_treated']}, units {e['n_units']}, "
             f"treatment period(s) {e['treated_times']}, matched-set size range {e['set_sizes']}, "
             f"{e['secs']}s)",
             f"{'lead':>5}{'ATT':>9}{'SE':>8}{'CI lo':>9}{'CI hi':>9}{'% (exp-1)':>11}"]
    
    for k, l in enumerate(e["lead"]):
        lines.append(f"{'t+' + str(l):>5}{e['estimate'][k]:9.4f}{e['se'][k]:8.4f}"
                     f"{e['ci_lo'][k]:9.4f}{e['ci_hi'][k]:9.4f}"
                     f"{100 * (np.exp(e['estimate'][k]) - 1):10.1f}%")
        
    return "\n".join(lines)


def save_att(e, out, name):
    save_json({k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in e.items() if k != "boots"}, os.path.join(out, name + ".json"))


def pm_main(c, out, say):
    e = estimate_att(c, "t1", "Fig 4 main (Mahalanobis, max matches)")
    say(format_att(e)); save_att(e, out, "fig4_main")
    plot_att(e, os.path.join(out, "fig4_main.png"), "", (-0.15, 0.5))

    e = estimate_att(c, "p1", "Fig A2 false date (2008-06-30)")
    say(format_att(e)); save_att(e, out, "figA2_false_date")
    plot_att(e, os.path.join(out, "figA2_false_date.png"), "Average Treatment Effect (False Date)", (-0.6, 0.15))

    e = estimate_att(c, "t1p", "Fig A4 pre-trend (2012)")
    say(format_att(e)); save_att(e, out, "figA4_pretrend")

    plot_att(e, os.path.join(out, "figA4_pretrend.png"), "Pretrend Test", (-0.3, 0.4), "Quarters before treatment administration")

    e = estimate_att(c, "t1", "Extra: covariates as presumably intended (rev, lt separate)", blocks=BLOCKS_INTENDED)
    say(format_att(e)); save_att(e, out, "extra_intended_covs")


def pm_robustness(c, out, say):
    specs = [("t1", "mahalanobis", 20, "A3a 20 max matches", (-0.15, 0.5)),
             ("t2", "mahalanobis", 4003, "A3b > median data-intensity", (-0.2, 0.9)),
             ("t5", "mahalanobis", 4003, "A3c alt. data-intensity (NAICS 'internet')", (-0.2, 0.7)),
             ("t1", "ps.match", 4003, "A3d PS matching", (-0.15, 0.5)),
             ("t1", "CBPS.match", 4003, "A3e CBPS matching (approx.)", (-0.15, 0.5)),
             ("t1", "ps.weight", 4003, "A3f PS weighting", (-0.15, 0.5))]
    
    for treatment, refinement, size, label, ylim in specs:
        e = estimate_att(c, treatment, label, refinement=refinement, size=size)
        say(format_att(e))
        key = label.split()[0]
        save_att(e, out, f"figA3_{key}")
        plot_att(e, os.path.join(out, f"figA3_{key}.png"), label, ylim)


def pm_placebo(c, out, say):
    rng = np.random.default_rng(2023)
    non_data = np.sort(c.loc[~c["naics"].isin(PAT_ALL), "gvkey"].unique())
    n_treated = c.loc[c["naics"].isin(PAT_ALL), "gvkey"].nunique()
    draws = []

    for i in range(30):
        sample = rng.choice(non_data, n_treated, replace=False)
        c["placebo2"] = (c["gvkey"].isin(sample) & (c["date"] > "2014-05-31")).astype(float)
        e = estimate_att(c, "placebo2", f"placebo draw {i + 1}", n_boot=1, seed=i)
        draws.append(e["estimate"])
        say(f"placebo {i + 1:2d}: " + " ".join(f"{v:7.3f}" for v in e["estimate"]))

    draws = np.array(draws)
    boot = np.array([[rng.choice(draws[:, j], 30).mean() for j in range(11)] for _ in range(500)])
    e = {"label": "Fig A5 placebo (30 draws of non-data-intensive firms)", "lead": list(range(11)),
         "estimate": draws.mean(0), "se": boot.std(0, ddof=1),
         "ci_lo": np.quantile(boot, .025, 0), "ci_hi": np.quantile(boot, .975, 0),
         "n_treated": n_treated, "n_units": None, "treated_times": None, "set_sizes": None,
         "secs": None}
    
    say(format_att(e))
    save_att(e, out, "figA5_placebo")
    plot_att(e, os.path.join(out, "figA5_placebo.png"), "Average Treatment Effect (Non-uptaking, 30 loops)", (-0.2, 0.2))

In [17]:
def run_firm_pm(data, work, out, part="all"):
    out = os.path.join(out, "firms_pm")
    os.makedirs(out, exist_ok=True)
    c, dates = chinese_firm_quarters(work)
    say = open_log(os.path.join(out, f"pm_{part}.txt"))
    say("Date index -> calendar date: " + ", ".join(f"{i + 1}={d}" for i, d in enumerate(dates)))
    say(f"Chinese firm-quarters {len(c):,}; unique data-intensive gvkeys (pat.di.all): "f"{c.loc[c['naics'].isin(PAT_ALL), 'gvkey'].nunique()}")
    
    if part in ("main", "all"):
        pm_main(c, out, say)
    if part in ("robust", "all"):
        pm_robustness(c, out, say)
    if part in ("placebo", "all"):
        pm_placebo(c, out, say)

In [18]:
def china_us_firm_dates(work):
    d = pd.read_csv(os.path.join(work, "firm_panel.csv"), dtype={"date": str, "loc": str})

    d["treat"] = d["naics"].isin(PAT_ALL).astype(float)
    d["treat2"] = d["naics"].isin(PAT_MED).astype(float)
    d["treat1"] = d["naics"].isin(NAICS_INTERNET).astype(float)
    d["china"] = (d["loc"] == "CHN").astype(float)
    d["censor"] = (d["date"] > "2014-05-31").astype(float)

    d = d[~d.duplicated(["date", "gvkey"])]

    for v in ["rev", "at", "lt", "che", "cstk", "seq", "lse"]:
        d.loc[d[v] < 0, v] = np.nan
        d["log" + v] = np.log(d[v] + 0.01)

    d["year"] = d["date"].str[:4]
    d = d[d["date"] > "2010-12-31"]

    return d[d["naics"].notna()]


def welch_t_test(a, b):
    va, vb = a.var() / len(a), b.var() / len(b)
    t = (a.mean() - b.mean()) / math.sqrt(va + vb)
    df = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
    return t, df, t_p_two_sided(t, df)


def summary_tables_2013(d, out, say):
    standard = d["date"].str.contains("03-31|06-30|09-30|12-31")
    cn13 = d[(d["china"] == 1) & (d["year"] == "2013") & standard]

    treated, control = cn13[cn13["treat"] == 1], cn13[cn13["treat"] == 0]

    cols = ["rev", "at", "che", "cstk", "lse", "lt", "seq", "logrev", "logat", "logche", "logcstk", "loglt", "logseq"]

    for name, df in [("A3_treated", treated), ("A4_control", control)]:
        s = df[cols].describe().T[["count", "mean", "std", "min", "max"]]
        s.to_csv(os.path.join(out, f"table{name}_summary_2013.csv"))
        say(f"\nTable {name[:2]} ({name[3:]}): Chinese firms 2013\n" + s.round(3).to_string())

    say("\nTable A5: Welch t-tests, data-intensive vs other Chinese firms (2013)")

    for v in ["logrev", "logat", "logche", "logcstk", "loglt", "logseq"]:
        a, b = treated[v].dropna(), control[v].dropna()
        t, df, p = welch_t_test(a, b)
        say(f"  {v:<8} treated {a.mean():8.3f}  control {b.mean():8.3f}  t = {t:7.3f}  "
            f"df = {df:7.1f}  p = {p:.4g}")


def triple_difference(d, treat_var, controls):
    x = d[["logrev", treat_var, "china", "censor", "year", "naics"] + controls].dropna()
    T, C, P = x[treat_var].to_numpy(), x["china"].to_numpy(), x["censor"].to_numpy()
    X = np.column_stack([T, C, P, T * C, T * P, C * P, T * C * P] + [x[c] for c in controls])
    names = [treat_var, "china", "censor", f"{treat_var}:china", f"{treat_var}:censor", "china:censor", f"{treat_var}:china:censor"] + controls
    return ols_fe(x["logrev"].to_numpy(), X, names, [x["year"].to_numpy()], x["naics"].to_numpy())

In [19]:
def run_ddd(data, work, out):
    out = os.path.join(out, "firms_ddd")
    os.makedirs(out, exist_ok=True)
    say = open_log(os.path.join(out, "ddd_results.txt"))

    d = china_us_firm_dates(work)
    say(f"DDD sample: {len(d):,} firm-dates ({d.groupby('loc')['gvkey'].nunique().to_dict()} firms), "f"{d['date'].min()} to {d['date'].max()}")
    
    summary_tables_2013(d, out, say)

    results = {}

    for label, controls in [("Table A7 (naive)", []), ("Table A8 (saturated: + log assets, log liabilities)", ["logat", "loglt"])]:
        say(f"\n================ {label} — NAICS-clustered SEs, year FE ================")

        for treat_var, name in [("treat", "patent-based, all"), ("treat2", "patent-based, > median"),
                                ("treat1", "NAICS 'internet'")]:
            r = triple_difference(d, treat_var, controls)
            say(f"\n{name}\n" + format_result(r))
            results[f"{label[:8]}:{name}"] = result_to_dict(r)

    save_json(results, os.path.join(out, "ddd_results.json"))

## Citations

In [20]:
def load_articles(work):
    read = dict(keep_default_na=False, na_values={"TC": [""], "auno": [""]},
                dtype={"SO": str, "area": str})
    rpp = pd.read_csv(os.path.join(work, "rpp.csv"))
    cn = pd.read_csv(os.path.join(work, "cn_articles.csv"), **read)
    us = pd.read_csv(os.path.join(work, "us_articles.csv"), **read)
    return rpp, cn, us


def figure_a1_knowledge_intensity(rpp, out):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(range(len(rpp)), np.log(rpp["rp"]), color="0.6")
    ax.set_xticks(range(len(rpp)))
    ax.set_xticklabels(rpp["area"], rotation=60, ha="right", fontsize=7)
    ax.set_ylabel("Logged references per page")
    fig.tight_layout()
    fig.savefig(os.path.join(out, "figA1_refs_per_page.png"), dpi=150)
    plt.close(fig)


def add_knowledge_variables(cn, rpp):
    cn = cn.merge(rpp[["area", "rp"]], on="area", how="left")
    cn["age"] = 2023 - cn["PY"]
    cn["t2"] = (cn["PY"] > 2014).astype(float)
    cn["rp"] = np.log(cn["rp"])
    med = cn["rp"].median()
    mean, sd = cn["rp"].mean(), cn["rp"].std()
    cn["rpdum1"] = np.where(cn["rp"] < med, 0.0, 1.0)
    cn["rpdum2"] = np.where(cn["rp"] < mean - sd, 0.0, np.where(cn["rp"] > mean + sd, 1.0, np.nan))
    return cn, med, mean, sd


def knowledge_model(df, ki, family):
    d = df[["TC", "t2", ki, "age", "auno", "area", "SO"]].dropna()
    X = np.column_stack([d["t2"], d[ki], d["age"], d["auno"], d["t2"] * d[ki]])
    names = ["t2", ki, "age", "auno", f"t2:{ki}"]
    return count_fe(d["TC"].to_numpy(float), X, names, [d["SO"].to_numpy()], d["area"].to_numpy(), family)


def figure_a6_parallel_trends(citeall, out, say):
    agg = citeall.groupby(["PY", "rpdum1"])["TC"].sum().unstack()
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(agg.index, np.log(agg[0.0]), color="blue", lw=2, label="Non-knowledge-intensive")
    ax.plot(agg.index, np.log(agg[1.0]), color="red", lw=2, label="Knowledge-intensive")
    ax.axvline(2014.5, ls="--", color="k")
    ax.text(2014.7, ax.get_ylim()[0] + .2, "2014 shock", weight="bold")
    ax.set_xlabel("Publication year")
    ax.set_ylabel("Logged total citations")
    ax.legend(loc="lower left")
    fig.tight_layout()
    fig.savefig(os.path.join(out, "figA6_parallel_trends.png"), dpi=150)
    plt.close(fig)
    say("\nFig A6 data (log total citations by year):\n" + np.log(agg).round(3).to_string())


def pretrend_test(citeall):
    pt = citeall[citeall["PY"] < 2015].dropna(subset=["rpdum1", "auno"])
    X = np.column_stack([pt["PY"], pt["rpdum1"], pt["age"], pt["auno"], pt["PY"] * pt["rpdum1"]])
    return ols_fe(np.log(pt["TC"].to_numpy(float) + 0.1), X, ["PY", "rpdum1", "age", "auno", "PY:rpdum1"], [pt["SO"].to_numpy()], pt["area"].to_numpy())


def china_us_model(uscn, family):
    d = uscn[["TC", "t2", "china", "age", "auno", "area", "SO"]].dropna()
    areas = sorted(d["area"].unique())
    dummies = np.column_stack([(d["area"] == a).to_numpy(float) for a in areas[1:]])
    X = np.column_stack([d["t2"], d["china"], d["age"], d["auno"], d["t2"] * d["china"], dummies])
    names = ["t2", "china", "age", "auno", "t2:china"] + [f"area:{a}" for a in areas[1:]]
    return count_fe(d["TC"].to_numpy(float), X, names, [d["SO"].to_numpy()], d["area"].to_numpy(), family, nuisance=len(areas) - 1)


def run_citations(data, work, out):
    out = os.path.join(out, "citations")
    os.makedirs(out, exist_ok=True)
    say = open_log(os.path.join(out, "citations_results.txt"))
    results = {}

    rpp, cn, us = load_articles(work)
    figure_a1_knowledge_intensity(rpp, out)
    cn, med, mean, sd = add_knowledge_variables(cn, rpp)
    citeall = cn[cn["TC"].notna()].copy()
    say(f"Chinese articles with citation counts (citeall): {len(citeall):,}")
    say(f"median log RP = {med:.4f}; mean = {mean:.4f}; sd = {sd:.4f}")
    say(f"rpdum2: 1 -> {sorted(citeall.loc[citeall.rpdum2 == 1, 'area'].unique())}")
    say(f"        0 -> {sorted(citeall.loc[citeall.rpdum2 == 0, 'area'].unique())}")

    field_year_median = citeall.groupby(["area", "PY"])["TC"].transform(lambda s: np.quantile(s, 0.5))
    cite50 = citeall[citeall["TC"] > field_year_median].copy()
    say(f"cite50 (above field-year median): {len(cite50):,}")

    a6 = citeall.groupby("area").agg(
        n=("TC", "size"), cites_mean=("TC", "mean"), cites_median=("TC", "median"),
        cites_max=("TC", "max"), age_mean=("age", "mean"), authors_mean=("auno", "mean"),
        authors_median=("auno", "median"))
    a6.to_csv(os.path.join(out, "tableA6_summary_by_discipline.csv"))
    say("\nTable A6 (summary by discipline)\n" + a6.round(2).to_string())

    specs = [("(1) RP continuous", citeall, "rp"), ("(2) RP, >50% cited", cite50, "rp"), ("(3) RP dummy >= median", citeall, "rpdum1"), ("(4) RP dummy +-1 SD", citeall, "rpdum2")]
    
    for family, label in [("negbin", "Table 1 (negative binomial)"), ("poisson", "Table A10 (Poisson)")]:
        say(f"\n================ {label} — discipline-clustered SEs ================")

        for name, df, ki in specs:
            r = knowledge_model(df, ki, family)
            say(f"\n{name}\n" + format_result(r, irr=True))
            results[f"{family}:{name}"] = result_to_dict(r)

    figure_a6_parallel_trends(citeall, out, say)
    r = pretrend_test(citeall)
    say("\n================ Table A9 (pre-trend OLS, log(TC+0.1)) ================\n" + format_result(r))
    results["A9"] = result_to_dict(r)

    us = us[us["TC"].notna()].copy()
    us["age"] = 2023 - us["PY"]
    us["t2"] = (us["PY"] > 2014).astype(float)
    us["china"] = 0.0

    citeall["china"] = 1.0

    cols = ["TC", "t2", "china", "age", "auno", "area", "SO"]
    uscn = pd.concat([us, citeall], ignore_index=True)[cols]

    say(f"\nUS-China pooled sample: {len(uscn):,} (US {len(us):,}, CN {len(citeall):,})")

    del us, cn, cite50

    say("================ Table 2 (US-China DiD), FE: discipline + journal ================")
    
    for family in ["negbin", "poisson"]:
        r = china_us_model(uscn, family)
        say(f"\n{family}\n" + format_result(r, irr=True))
        results[f"T2:{family}"] = result_to_dict(r)
    save_json(results, os.path.join(out, "citations_results.json"))

In [21]:
RUN = ["prep", "figures", "firm_pm", "ddd", "citations"]

PM_PART = "all"
DATA_DIR = DEFAULT_DATA
WORK_DIR = DEFAULT_WORK
OUT_DIR = DEFAULT_OUT

STEPS = {
    "prep":      lambda: run_prep(DATA_DIR, WORK_DIR, OUT_DIR),
    "figures":   lambda: run_figures(DATA_DIR, WORK_DIR, OUT_DIR),
    "firm_pm":   lambda: run_firm_pm(DATA_DIR, WORK_DIR, OUT_DIR, PM_PART),
    "ddd":       lambda: run_ddd(DATA_DIR, WORK_DIR, OUT_DIR),
    "citations": lambda: run_citations(DATA_DIR, WORK_DIR, OUT_DIR),
}

if __name__ == "__main__":
    for name in RUN:
        STEPS[name]()

== Knowledge intensity and articles
RPP rows (R: nrow(rpp)): 4825
RPP disciplines: 31
                                              refs  pages        rp
area                                                               
Agriculture                                   8521   1552  5.490335
Astronomy & Astrophysics                      9920   2087  4.753234
Biochemistry & Molecular Biology              7107   1624  4.376232
Business & Economics                         17260   5062  3.409719
Cardiovascular System & Cardiology           11752   2134  5.507029
Cell Biology                                  3094    714  4.333333
Chemistry                                    28816   3761  7.661792
Computer Science                              7239   2424  2.986386
Endocrinology & Metabolism                    5374    912  5.892544
Engineering                                  13320   3729  3.572003
Environmental Sciences & Ecology              8009   1514  5.289960
Gastroenterology & Hepatology 

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/_core/_methods.py:208: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


placebo  2:  -0.003  -0.042  -0.015  -0.070  -0.059  -0.112  -0.057  -0.062  -0.049  -0.046  -0.060


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/_core/_methods.py:208: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


placebo  3:   0.003   0.009   0.013   0.010  -0.030   0.011   0.038   0.071   0.035   0.025   0.016


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/_core/_methods.py:208: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


... (51 outputs left out)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/_core/_methods.py:208: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


placebo 30:   0.047   0.055   0.035   0.069   0.065   0.075   0.059   0.029   0.080   0.094   0.011
--- Fig A5 placebo (30 draws of non-data-intensive firms)  (treated obs 216, units None, treatment period(s) None, matched-set size range None, Nones)
 lead      ATT      SE    CI lo    CI hi  % (exp-1)
  t+0   0.0036  0.0060  -0.0082   0.0151       0.4%
  t+1  -0.0002  0.0092  -0.0187   0.0176      -0.0%
  t+2  -0.0100  0.0118  -0.0308   0.0149      -1.0%
  t+3   0.0001  0.0082  -0.0156   0.0168       0.0%
  t+4   0.0080  0.0115  -0.0140   0.0316       0.8%
  t+5   0.0019  0.0151  -0.0249   0.0332       0.2%
  t+6  -0.0082  0.0171  -0.0429   0.0256      -0.8%
  t+7  -0.0091  0.0112  -0.0297   0.0134      -0.9%
  t+8  -0.0041  0.0153  -0.0329   0.0255      -0.4%
  t+9  -0.0037  0.0169  -0.0370   0.0288      -0.4%
 t+10  -0.0253  0.0134  -0.0497   0.0023      -2.5%


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/_core/_methods.py:208: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


DDD sample: 145,305 firm-dates ({'CHN': 4454, 'USA': 6542} firms), 2011-01-31 to 2017-03-31

Table A3 (treated): Chinese firms 2013
         count       mean        std    min         max
rev      221.0   1368.764   9231.514  0.000   95047.059
at       251.0  11746.782  66190.673  1.766  555083.998
che      251.0   1295.753  10231.823  0.000  138042.519
cstk     249.0    378.911   2717.886  0.000   30232.915
lse      251.0  11746.782  66190.673  1.766  555083.998
lt       251.0   8961.626  54356.486  1.197  515487.553
seq      240.0   2627.222  17268.552  0.112  187114.492
logrev   221.0      3.891      1.992 -4.605      11.462
logat    251.0      5.780      1.852  0.574      13.227
logche   251.0      4.215      1.854 -4.605      11.835
logcstk  249.0      3.165      2.528 -4.605      10.317
loglt    251.0      4.574      2.215  0.188      13.153
logseq   240.0      5.285      1.642 -2.108      12.139

Table A4 (control): Chinese firms 2013
          count      mean        std    min 